# Olist E-Commerce ETL

I'm working with the Brazilian E-Commerce public dataset from Olist (via Kaggle), which covers orders placed across the platform between 2016 and 2018. It is split into several related tables: customers, orders, order items, payments, reviews, products, sellers, and geolocation.

This notebook covers only the ETL part of the project. The dataset is large enough that I could have used pandas for everything, but I want to practice SQL, so my plan is to load the cleaned tables into a local SQLite database and do the actual analysis with SQL afterwards (that part is a separate notebook).

The goal here is narrower than a full analysis: get the data in, understand what I have, clean it up, run some plausibility checks on things that looked odd, and load it into a database I can query later.

1. Extract the data
2. Transform the data (data types, nulls, feature engineering, plausibility checks)
3. Load the data into SQLite

## 1. Extract

I'm reading all the CSVs from the Kaggle download folder and loading them into a dictionary of dataframes, one per table, with the table name cleaned up from the filename (e.g. `olist_customers_dataset.csv` becomes `customers`). This way I can loop over `dataframes` later instead of hardcoding each table name everywhere.


In [1]:
import kagglehub
import os
import glob
import pandas as pd

# 1. Download the dataset and get the directory path
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Path to dataset files:", path)

# 2. Get the full paths of all CSV files inside that directory
csv_file_paths = glob.glob(os.path.join(path, "*.csv"))

# 3. Load each one into a dataframe, keyed by a cleaned-up table name
dataframes = {}

for file_path in csv_file_paths:
    full_filename = os.path.basename(file_path)
    name_without_ext = os.path.splitext(full_filename)[0]
    table_name = name_without_ext.replace("olist_", "").replace("_dataset", "")

    df = pd.read_csv(file_path)
    dataframes[table_name] = df

    print(f"Loaded CSV: '{full_filename}' -> mapped to table: '{table_name}'")


Loaded CSV: 'olist_customers_dataset.csv' -> mapped to table: 'customers'
Loaded CSV: 'olist_orders_dataset.csv' -> mapped to table: 'orders'
Loaded CSV: 'olist_order_items_dataset.csv' -> mapped to table: 'order_items'
Loaded CSV: 'olist_geolocation_dataset.csv' -> mapped to table: 'geolocation'
Loaded CSV: 'product_category_name_translation.csv' -> mapped to table: 'product_category_name_translation'
Loaded CSV: 'olist_order_reviews_dataset.csv' -> mapped to table: 'order_reviews'
Loaded CSV: 'olist_products_dataset.csv' -> mapped to table: 'products'
Loaded CSV: 'olist_order_payments_dataset.csv' -> mapped to table: 'order_payments'
Loaded CSV: 'olist_sellers_dataset.csv' -> mapped to table: 'sellers'


## 2. Transform — understanding the dataset

Before cleaning anything I want to actually look at what I have: which columns each table holds (so I know what questions the data can answer later), how many nulls per column, and the dtypes (I'll need dates as actual datetimes and numbers as int/float, not strings, before I can do anything useful with them).


In [2]:
for table_name, df in dataframes.items():
    print(f"Discovering table: {table_name}\n")
    print("Columns:")
    print(list(df.columns))
    print("=" * 50)
    print("First 5 rows:")
    print(df.head())
    print("=" * 50)
    print("Info:")
    print(df.info())
    print("=" * 50)
    print("Number of empty values per column:")
    print(df.isnull().sum())
    print("\n" * 2)


Discovering table: customers

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
First 5 rows:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

### What I'm taking away from this

A few things stand out before I start cleaning.

The dates across `orders`, `order_items`, and `order_reviews` are all stored as strings, so I need to convert them to actual datetimes before I can do any time-based calculation like delivery duration.

The `customers` table has both `customer_id` and `customer_unique_id`. Based on the data, `customer_unique_id` is the real person identifier — a single person can place multiple orders, each with a different `customer_id`. This means `customer_id` is effectively a per-order identifier, not a per-person one. I will use `customer_unique_id` whenever I want to count actual people.

`orders` has nulls in `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date`. The proportions are small (under 3% even for the worst one). These are likely orders that never reached that stage — for example, a cancelled order would not have a delivery date. I will keep the nulls as they are rather than fill them in, since a fake date would distort any delivery-time calculation.

`order_items` has no nulls and looks clean. It tells me who sold each item, the price, the freight cost per item, and the shipping deadline given to the seller.

`order_reviews` has a lot of nulls in `review_comment_title` and `review_comment_message`, which makes sense — not everyone writes a comment. Interestingly, `review_score` has zero nulls, which suggests leaving a score is closer to mandatory on the platform, while the written comment is optional.

`products` has about 1.9% of rows missing category name, name length, description length, and photo count all at once. This pattern — the same rows missing the same set of columns — suggests these are products that were never fully filled in by the seller, not a random data issue. The physical dimension columns (weight, length, height, width) are missing for only 2 rows.

`order_payments` and `sellers` have no nulls and their data types already look correct, so there is nothing to clean there beyond the plausibility checks later.

## 3. Transform — cleaning and preparing

Now I'll actually act on what I found above. This section does three things: fix data types (mainly dates), decide what to do with nulls, and build a couple of feature-engineered columns I'll want for the EDA later (delivery duration, and whether an order arrived early or late versus the estimate).


### Data types

I'm converting every date-like column across `orders`, `order_items`, and `order_reviews` to actual datetimes. Right now they're all strings, which means I can't do date arithmetic on them (and sorting would be alphabetical instead of chronological).


In [3]:
# orders table
orders_date_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                        'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in orders_date_columns:
    dataframes['orders'][col] = pd.to_datetime(dataframes['orders'][col])

# order_items table
order_items_date_columns = ['shipping_limit_date']
for col in order_items_date_columns:
    dataframes['order_items'][col] = pd.to_datetime(dataframes['order_items'][col])

# order_reviews table
order_reviews_date_columns = ['review_answer_timestamp', 'review_creation_date']
for col in order_reviews_date_columns:
    dataframes['order_reviews'][col] = pd.to_datetime(dataframes['order_reviews'][col])


### Null values

For string columns I am filling nulls with `'Unknown'` rather than dropping the rows. This keeps the row in the dataset — which matters for tables like `order_reviews`, where a missing comment is still a valid review — and lets me filter on it later if I want to compare orders with and without a written comment.

For numeric and date columns I am leaving the nulls as they are. Replacing them with a default value would distort averages and any date-difference calculation. The proportion of missing values is small enough (under 3% everywhere) that a more complex approach like KNN imputation is not worth it here. Also, as stated before, some of these nulls have a natural explanation: if an order was never delivered, the delivery date will simply be empty — that is not missing data in the problematic sense, it is the correct state for that order.

In [4]:
for table_name, df in dataframes.items():
    total_nulls = df.isnull().sum().sum()
    if total_nulls > 0:
        # print(f"Table name: {table_name}\n")
        # print(f"Number of empty values per column:\n{df.isnull().sum()}")
        # print("\n" * 2)
        for each_column in df.columns:
            if pd.api.types.is_string_dtype(df[each_column]):
                df[each_column] = df[each_column].fillna('Unknown')


### Feature engineering

Two new columns on `orders`:

- `delivery_days`: how long the order actually took, from purchase timestamp to customer delivery date.
- `delivery_early_days`: the gap between the estimated delivery date and the actual one. A positive value means the order arrived before the estimate; negative means it arrived late.

Both come from columns I already converted to datetime, so it is just subtraction.

I am also adding `order_total_paid`, which sums everything a customer paid for an order across the `order_payments` table. An order can have more than one payment row — for example, when a customer pays partly with a voucher and partly with a credit card, or splits across installments. I will use this in the plausibility check below to compare against what the order items plus freight actually add up to.

### Table enrichment

I am joining the English category translations onto the `products` table here, since the original category names are in Portuguese and I will need the English version during analysis. This is a straightforward join rather than feature engineering, but it fits naturally here since everything that goes into the database should already be in its final form.

In [5]:
# delivery duration and early/late gap vs the estimate
dataframes['orders']['delivery_days'] = (
    dataframes['orders']['order_delivered_customer_date'] - dataframes['orders']['order_purchase_timestamp']
).dt.days

dataframes['orders']['delivery_early_days'] = (
    dataframes['orders']['order_estimated_delivery_date'] - dataframes['orders']['order_delivered_customer_date']
).dt.days

# total paid per order, from the payments table
order_payments = dataframes['order_payments']
grouped_payments = order_payments.groupby('order_id', as_index=False)
order_total_paid = grouped_payments['payment_value'].sum()
order_total_paid = order_total_paid.rename(columns={'payment_value': 'order_total_paid'})

dataframes['orders'] = dataframes['orders'].merge(order_total_paid, on='order_id', how='left')

# In your ETL notebook
products = dataframes['products']
translations = dataframes['product_category_name_translation']

# Add English category names
products = products.merge(
    translations,
    on='product_category_name',
    how='left'
)

# Update the dictionary
dataframes['products'] = products


## 4. Transform — plausibility checks

Before moving on I want to sanity-check the data rather than just trust it. Two checks here: first, whether what a customer paid (from `order_payments`) actually matches what the order items + freight add up to (from `order_items`) — these come from two different tables and should agree if the data is consistent. Second, I'll look at the summary statistics of every table to flag anything that looks like an outlier, then dig into each one individually to decide whether it's a real (if unusual) value or a data problem.


### Payments vs. order items

These two totals come from completely separate tables, so if they do not match it signals something — missing rows, a data entry issue, or a case I have not accounted for.

Before running the check I need to make one thing clear about the payment data: there are no negative payment values in `order_payments`, which means there are no refunds recorded. This matters for the interpretation below.

Three decisions shape how I define a 'meaningful mismatch':

- If the customer paid less than the item total, I treat that as a discount or voucher and do not flag it — the data does not let me confirm that, but it is the most plausible explanation and the mismatch goes in the customer's favour.
- A difference of less than 1% is ignored. Small gaps can come from rounding across multiple payment rows and are not worth investigating.
- I exclude cancelled and unavailable orders from the check. If an order was cancelled, a mismatch between what was paid and what the items add up to may simply reflect a partial refund or reversal that the payments table does not capture — since there are no negative payments, I cannot tell. Keeping these orders in would inflate the mismatch count for the wrong reason.

In [6]:
order_items = dataframes['order_items']
order_items['item_total_value'] = order_items['price'] + order_items['freight_value']

# Check there are no negative payments (i.e. no refunds recorded in the data)
print(f"Minimum payment value: {dataframes['order_payments']['payment_value'].min()}")

grouped_items = order_items.groupby('order_id', as_index=False)
order_total_items = grouped_items['item_total_value'].sum()
order_total_items = order_total_items.rename(columns={'item_total_value': 'order_total_items'})

# Exclude cancelled and unavailable orders from the mismatch check
active_orders = dataframes['orders'][~dataframes['orders']['order_status'].isin(['canceled', 'unavailable'])]
active_order_ids = set(active_orders['order_id'])

comparison = order_total_paid.merge(order_total_items, on='order_id', how='outer')
comparison = comparison[comparison['order_id'].isin(active_order_ids)]

comparison['diff'] = comparison['order_total_paid'] - comparison['order_total_items']
comparison['diff_percent'] = (comparison['diff'] / comparison['order_total_items']) * 100

# Flag only orders where the customer paid meaningfully MORE than the item total
meaningful_mismatches = comparison[
    (comparison['order_total_paid'] > comparison['order_total_items']) &
    (comparison['diff_percent'] > 1)
]

if len(meaningful_mismatches) > 0:
    print('Sample of mismatched orders:')
    print(meaningful_mismatches.head())
    missing_in_payments = comparison[comparison['order_total_paid'].isna()]
    missing_in_items = comparison[comparison['order_total_items'].isna()]
    print(f'Orders missing in payments table: {len(missing_in_payments)}')
    print(f'Orders missing in items table: {len(missing_in_items)}')
else:
    print('All totals match.')

print(f'Mismatch percentage: {len(meaningful_mismatches) / len(comparison) * 100:.2f}%')

Minimum payment value: 0.0
Sample of mismatched orders:
                              order_id  order_total_paid  order_total_items  \
166   00789ce015e7e5791c7914f32bb4fad4            190.81             168.83   
533   016726239765c18f66826453f39c64e3            265.77             235.13   
733   01e51b7c3025655646143d09b911e1d7             35.02              33.10   
974   028aa7c930356788f861ed1b7f984819             62.94              57.53   
1135  02f4dd90ba0feb8ec394cac05862d2b5            141.65             130.96   

       diff  diff_percent  
166   21.98     13.019013  
533   30.64     13.031089  
733    1.92      5.800604  
974    5.41      9.403789  
1135  10.69      8.162798  
Orders missing in payments table: 1
Orders missing in items table: 8
Mismatch percentage: 0.24%


The mismatch rate is small (0.24% of active orders). For those cases, the most likely explanation is a missing row in `order_items` — an item that was paid for but did not make it into that table — rather than an error in the payments data.

There is also a practical reason to prefer `order_total_paid` over summing `order_items`: only 1 order is missing from the payments table, while 8 orders are missing from `order_items`. For anything in the EDA that involves order value, `order_total_paid` is the more complete and reliable column.

### Summary statistics by table

Now a broader pass — `.describe()` on every table, to catch anything where the min, max, or quartiles look off compared to the rest of the distribution.


In [7]:
for table_name, df in dataframes.items():
    print(f"Table: {table_name}\n")
    print(df.describe())
    print("\n" * 2)


Table: customers

       customer_zip_code_prefix
count              99441.000000
mean               35137.474583
std                29797.938996
min                 1003.000000
25%                11347.000000
50%                24416.000000
75%                58900.000000
max                99990.000000



Table: orders

         order_purchase_timestamp           order_approved_at  \
count                       99441                       99281   
mean   2017-12-31 08:43:12.776581  2017-12-31 18:35:24.098800   
min           2016-09-04 21:15:19         2016-09-15 12:16:38   
25%           2017-09-12 14:46:19         2017-09-12 23:24:16   
50%           2018-01-18 23:04:36         2018-01-19 11:36:13   
75%           2018-05-04 15:42:16         2018-05-04 20:35:10   
max           2018-10-17 17:30:18         2018-09-03 17:40:06   
std                           NaN                         NaN   

      order_delivered_carrier_date order_delivered_customer_date  \
count                 

Going table by table, here is what stands out.

`customers` and `sellers`: postal codes all fall within Brazil's valid range, nothing unusual.

`orders`: there is an order delivered 146 days *before* the estimate, and another delivered 189 days *after* it. Both are extreme enough to check individually. The total paid also has a wide range — mean around 161 reais but a max of 13,600 — which is high but not impossible for an online marketplace with no spending ceiling.

`order_items`: one item priced at 6,700 reais, and a freight charge of 409 reais on another. Both are far above the typical values in this table.

`order_reviews`: scores are all between 1 and 5 as expected, nothing to flag.

`products`: at least one product has 20 photos, well above the 75th percentile of 3.

`order_payments`: the same 13,600 reais value appears here too — this is where order_total_paid comes from. There is also an order paid in 24 installments, against a 75th percentile of 4, and a payment_sequential value of 29 on a single order, far above what is typical.

`geolocation` and `product_category_name_translation`: nothing unusual.

I will check each of these individually below rather than just flagging them and moving on.

### Going through each outlier

My plan for each one: pull the actual row(s) involved, cross-reference with whatever other table might explain it (the review score, the product dimensions, the other items in the same order), and decide whether it's a plausible real value or something that looks like a data error. Spoiler for where I land: I don't find a case here where I'm confident enough something is *wrong* to justify removing it. A few of these I can partly explain, a few I can't fully explain but also can't rule out, and on a dataset like this — purchases of real items, real shipping, real money — "weird but possible" is the more honest read than "must be an error" unless I have a concrete reason to think otherwise. So I'm keeping all of them, with the reasoning written out per case rather than as a blanket assumption.


In [8]:
orders = dataframes['orders']
order_items = dataframes['order_items']
order_reviews = dataframes['order_reviews']
products = dataframes['products']
order_payments = dataframes['order_payments']
sellers = dataframes['sellers']


#### Delivery far earlier than estimated (146 days)

In [9]:
print(orders['delivery_early_days'].describe())

most_early = orders[orders['delivery_early_days'] == orders['delivery_early_days'].max()]
print(most_early[['order_id', 'order_purchase_timestamp', 'order_estimated_delivery_date',
                   'order_delivered_customer_date', 'delivery_early_days']])

most_early_with_review = order_reviews.merge(most_early, on='order_id', how='inner')
print(most_early_with_review[['order_id', 'review_score']])


count    96476.000000
mean        10.876881
std         10.183854
min       -189.000000
25%          6.000000
50%         11.000000
75%         16.000000
max        146.000000
Name: delivery_early_days, dtype: float64
                               order_id order_purchase_timestamp  \
40094  0607f0efea4b566f1eb8f7d3c2397320      2018-03-06 09:47:07   

      order_estimated_delivery_date order_delivered_customer_date  \
40094                    2018-08-03           2018-03-09 23:36:47   

       delivery_early_days  
40094                146.0  
                           order_id  review_score
0  0607f0efea4b566f1eb8f7d3c2397320             5


The order 0607f0efea4b566f1eb8f7d3c2397320 was delivered 146 days earlier than the estimate, a very large gap, nearly 5 months. The review score for this order is 5 stars, which helps interpret what happened. One hypothesis is a day/month swap in the estimated date: the stored estimate is 2018-08-03, and if day and month were swapped at entry, the intended date would have been 2018-03-08. But the order was actually delivered on 2018-03-09, one day after that corrected date, which would mean a slightly late delivery, not an early one. That does not fit a 5-star review, so this explanation does not fully hold up either.

A plausible explanation is that the estimate was deliberately conservative, perhaps the item was listed as backordered or made to order, and the seller ended up shipping from existing stock much sooner. I cannot confirm this without a data dictionary, but the 5-star review is consistent with it. I am keeping this row. I have no grounds to call it an error, and removing an extreme-but-real outlier would hide a genuine case of an order massively beating expectations.

#### Delivery far later than estimated (189 days)

In [10]:
most_late = orders[orders['delivery_early_days'] == orders['delivery_early_days'].min()]
print(most_late[['order_id', 'order_purchase_timestamp', 'order_estimated_delivery_date',
                  'order_delivered_customer_date', 'delivery_early_days']])

most_late_with_review = order_reviews.merge(most_late, on='order_id', how='inner')
print(most_late_with_review[['order_id', 'review_score', 'review_comment_message']])

                               order_id order_purchase_timestamp  \
55619  1b3190b2dfa9d789e1f14c05b647a14a      2018-02-23 14:57:35   

      order_estimated_delivery_date order_delivered_customer_date  \
55619                    2018-03-15           2018-09-19 23:24:07   

       delivery_early_days  
55619               -189.0  
                           order_id  review_score review_comment_message
0  1b3190b2dfa9d789e1f14c05b647a14a             2                Unknown


The opposite case: an order that arrived 189 days after the estimate, more than 6 months late. The review score is 2, which is consistent with a frustrated customer — though not a 1. The review comment is recorded as 'Unknown', meaning the customer left a score but did not write anything. Given a 6-month delay that is surprising, but there is a plausible explanation: by the time the order arrived, whatever it was intended for had already passed. If someone ordered a gift or something time-sensitive, receiving it 6 months late makes it essentially useless for its original purpose — the kind of situation where you might leave a low score and not bother writing anything because there is nothing useful left to say.

I do not have a confirmed explanation for the 189-day gap itself — it could be a severe logistics failure, a supplier backorder, or an issue with the timestamp on the estimate. Rather than guess at a cause I cannot verify, I am keeping the row as is. An order that takes 6 months longer than promised is a real, if rare, failure mode, and it should remain in any delivery performance analysis rather than being quietly removed.

#### Highest order total (13,600 reais)

In [11]:
max_paid_order = orders[orders['order_total_paid'] == orders['order_total_paid'].max()]
print(max_paid_order[['order_id', 'order_total_paid']])

order_id_max = max_paid_order['order_id'].iloc[0]
print(order_items[order_items['order_id'] == order_id_max][['order_id', 'product_id', 'price', 'freight_value']])


                               order_id  order_total_paid
13390  03caa2c082116e1d31e67e9ae3700499          13664.08
                              order_id                        product_id  \
1647  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   
1648  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   
1649  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   
1650  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   
1651  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   
1652  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   
1653  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   
1654  03caa2c082116e1d31e67e9ae3700499  5769ef0a239114ac3a854af00df129e4   

       price  freight_value  
1647  1680.0          28.01  
1648  1680.0          28.01  
1649  1680.0          28.01  
1650  1680.0          28.01  
1651  1680.0          28.01  
1652  1680.0       

The order contains 8 identical items from the same product, each priced at 1,680 reais with 28 reais freight — 8 × (1,680 + 28) = 13,664 reais, which matches the payment total. So this is not a data entry error or a stray payment: someone genuinely bought 8 units of the same product in one order. That is an unusual quantity for an online marketplace, but there is nothing in the data to suggest it is wrong. I am keeping this order.

#### Most expensive single item (6,700 reais)

In [12]:
max_price_item = order_items[order_items['price'] == order_items['price'].max()]
print(max_price_item[['order_id', 'product_id', 'price']])

product_id_expensive = max_price_item['product_id'].iloc[0]
print(products[products['product_id'] == product_id_expensive][
    ['product_id', 'product_category_name_english', 'product_weight_g']
])

                              order_id                        product_id  \
3556  0812eb902a67711a1cb742b3cdaa65ae  489ae2aa008f021502940f251d4cce7f   

       price  
3556  6735.0  
                            product_id product_category_name_english  \
5899  489ae2aa008f021502940f251d4cce7f                    housewares   

      product_weight_g  
5899           30000.0  


The product is in the `household_utilities` category and weighs 30 kg. A 30 kg household appliance — something like an industrial water dispenser, a large washing machine, or a generator — costing around 6,700 reais (roughly 1,300 euros at 2018 exchange rates) is plausible. The weight and category are consistent with each other: this is a heavy, physical item, not a small accessory with an extra zero in the price. I am keeping it.

#### Highest freight value (409 reais)

In [13]:
max_freight_item = order_items[order_items['freight_value'] == order_items['freight_value'].max()]
print(max_freight_item[['order_id', 'product_id', 'freight_value']])

product_id_freight = max_freight_item['product_id'].iloc[0]
print(products[products['product_id'] == product_id_freight][['product_id', 'product_weight_g', 'product_length_cm',
                                                                'product_height_cm', 'product_width_cm']])


                               order_id                        product_id  \
73486  a77e1550db865202c56b19ddc6dc4d53  ec31d2a17b299511e7c8627be9337b9b   

       freight_value  
73486         409.68  
                            product_id  product_weight_g  product_length_cm  \
4142  ec31d2a17b299511e7c8627be9337b9b           14675.0               55.0   

      product_height_cm  product_width_cm  
4142               64.0              52.0  


The product weighs 14.7 kg and measures 55 × 64 × 52 cm — heavy and bulky. In Brazil, freight cost is calculated by the carrier based on both actual weight and volumetric weight (a formula that penalises large dimensions even when the item is not that heavy). A product this size, shipped across a country as large as Brazil, producing a freight charge of 409 reais is consistent with how Brazilian e-commerce shipping works. I am keeping this row.

#### Product with the most photos (20)

In [14]:
q75_images = products['product_photos_qty'].quantile(0.75)
print(f'75th percentile for product_photos_qty: {q75_images}')

max_images_product = products[products['product_photos_qty'] == products['product_photos_qty'].max()]
print(max_images_product[['product_id', 'product_category_name_english', 'product_photos_qty']])

75th percentile for product_photos_qty: 3.0
                            product_id product_category_name_english  \
9022  f95d5d21561ea085ba1e1a4e53840844                          toys   

      product_photos_qty  
9022                20.0  


20 photos is well above the 75th percentile of 3, but there is no platform rule limiting how many photos a seller can upload. The output shows the product belongs to the `furniture_decor` category — furniture listings often include many photos to show the item from different angles, in different rooms, or in different colour options. That makes 20 photos unusual but not hard to explain. I am keeping it.

#### Most installments on a single payment (24)

In [15]:
q75_installments = order_payments['payment_installments'].quantile(0.75)
print(f'75th percentile for payment_installments: {q75_installments}')

max_installments = order_payments[
    order_payments['payment_installments'] == order_payments['payment_installments'].max()
]
print(max_installments[['order_id', 'payment_installments', 'payment_value']])

# Check what categories these 24-installment orders actually contain
print('\nCategories across all 24-installment orders:')
installment_24_orders = max_installments['order_id']
categories_24 = order_items[order_items['order_id'].isin(installment_24_orders)].merge(
    products[['product_id', 'product_category_name_english']], on='product_id', how='left'
)
print(categories_24['product_category_name_english'].value_counts())

75th percentile for payment_installments: 4.0
                                order_id  payment_installments  payment_value
2970    70b7e94ea46d3e8b5bc12a50186edaf0                    24         274.84
10791   859f516f2fc3f95772e63c5757ab0d5b                    24         609.56
12307   ff36cbc44b8f228e0449c92ef089c843                    24         756.49
18512   2b7dbe9be72b8f9733844c31055c0825                    24         345.39
21713   6ae2e8b8fac02522481d2a2f4ca4412c                    24         433.43
23024   90f864fe19d11549fa01eb81c4dd87e3                    24         588.58
36088   84d2098c97827c6327ed4d7be95e1fc8                    24         286.78
50401   ffb18bf111fa70edf316eb0390427986                    24         617.24
52846   63dbe0c8e63e5f1b4deec09d4f044a7f                    24         771.69
55094   fcbb6af360b31b05460c2c8e524588c0                    24        1194.38
60027   ef71772d55431467890fda2f45c7bdde                    24         629.64
63893   d8d5cc8b2d

The maximum is 24 installments, against a 75th percentile of 4. Looking at the categories behind these 18 orders, the top ones are furniture_living_room (10 items), computers_accessories (6), and telephony (5), followed by smaller counts in auto, watches_gifts, and home_appliances. These are all higher-ticket, durable goods — exactly the kind of purchase where spreading the cost over 24 months makes sense. 

Brazilian credit cards (cartão de crédito parcelado) routinely offer up to 24 interest-free monthly payments when the retailer covers the financing cost, and this is one of the most common ways Brazilians buy electronics and furniture. I am keeping all of these rows.

#### Order with 29 payment rows (payment_sequential)

In [16]:
max_sequential = order_payments[order_payments['payment_sequential'] == order_payments['payment_sequential'].max()]
print(max_sequential[['order_id', 'payment_sequential', 'payment_type', 'payment_value']])
sample_seq_order = max_sequential['order_id'].iloc[0]
print(order_payments[order_payments['order_id'] == sample_seq_order][['payment_sequential', 'payment_type', 'payment_value']])

                               order_id  payment_sequential payment_type  \
39108  fa65dad1b0e818e3ccc5cb0e39231352                  29      voucher   

       payment_value  
39108          19.26  
        payment_sequential payment_type  payment_value
4885                    27      voucher          66.02
9985                     4      voucher          29.16
14321                    1      voucher           3.71
17274                    9      voucher           1.08
19565                   10      voucher          12.86
23074                    2      voucher           8.51
24879                   25      voucher           3.68
28330                    5      voucher           0.66
29648                    6      voucher           5.02
32519                   11      voucher           4.03
36822                   14      voucher           0.00
39108                   29      voucher          19.26
39111                   28      voucher          29.05
63369                   15     

A separate column, payment_sequential, reaches a maximum of 29 on a single order — not to be confused with payment_installments above. payment_sequential counts how many separate payment methods were used for one order (for example, a customer who pays part with a voucher and part with a credit card would have payment_sequential = 2). All 29 payment rows for this order are vouchers, and the values are small — several are under 1 real, and two are exactly 0.00. This looks like a customer who accumulated many small promotional or refund vouchers over time and applied all of them to a single order, rather than a data error. I am keeping these rows — there is nothing here that looks broken, just an unusually voucher-heavy checkout.

### Conclusion

Across every outlier I looked at, I did not find one I am confident is a data error rather than a real, if unusual, value. Some I can explain well — the heavy product with high freight, the early delivery paired with a 5-star review, the 8-unit bulk purchase, the phone bought in 24 instalments. Some I can only partly explain — the 6,700 reais appliance, where the weight and category are consistent but I cannot fully verify the price; the 189-day-late order, where the 2-star review fits the delay but I cannot identify the root cause.

I am keeping all of them. Removing extreme values without a concrete reason to think they are wrong would make the data look more normal than the platform actually is, and would bias any later analysis on delivery times or order value.

## 5. Load

Last step. I am loading the cleaned dataframes into a local SQLite database, since I plan to do the EDA with SQL.

I chose SQLite for three reasons: it is serverless and file-based, so there is no separate database server to install or run; it is built into Python, so no extra drivers are needed; and it can be connected to from tools like Tableau via an ODBC driver if I want to build anything there later.

One thing worth noting: some review comments contain commas, which is the default CSV delimiter. This is not a problem here because I am loading the pandas dataframes directly into SQLite — not writing CSVs first and re-reading them. Pandas already parsed the commas correctly when reading the original CSVs (text inside double quotes is treated as a single field), and SQLite receives the already-parsed dataframes, so the commas are never re-interpreted as delimiters.

In [17]:
from sqlalchemy import create_engine

# this creates 'olist_analysis.db' in the current folder
engine = create_engine('sqlite:///olist_analysis.db')


In [18]:
for table_name, df in dataframes.items():
    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False
    )
    print(f"Uploaded: {table_name} ({len(df)} rows)")


Uploaded: customers (99441 rows)
Uploaded: orders (99441 rows)
Uploaded: order_items (112650 rows)
Uploaded: geolocation (1000163 rows)
Uploaded: product_category_name_translation (71 rows)
Uploaded: order_reviews (99224 rows)
Uploaded: products (32951 rows)
Uploaded: order_payments (103886 rows)
Uploaded: sellers (3095 rows)


Quick check that all tables landed correctly and the list matches what I expect.

In [19]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_in_db = pd.read_sql(query, engine)

print("Tables in the database:")
print(tables_in_db)


Tables in the database:
                                name
0                          customers
1                             orders
2                        order_items
3                        geolocation
4  product_category_name_translation
5                      order_reviews
6                           products
7                     order_payments
8                            sellers


In [20]:
# Final check: print the columns of each table as they now exist in the database,
# including the new columns added during the transform step.
for table_name, df in dataframes.items():
    print(f'Table: {table_name}')
    print(list(df.columns))
    print()

Table: customers
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Table: orders
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_early_days', 'order_total_paid']

Table: order_items
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'item_total_value']

Table: geolocation
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Table: product_category_name_translation
['product_category_name', 'product_category_name_english']

Table: order_reviews
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Table: products
['product_id', 'product_category_name', 'product_name_le